# Data Mining and Preparation on a Flat File Source
Handling sensitive data, ethically. 

https://datahub.thetrace.org/dataset/school-adjacent-shootings/#details

## Code Attribution
<ul>
  <li> Title: Python for Data Analysis 
  <li> Author: Wes McKinney
  <li> Date: 19 October 2025
  <li> Modified By: Charlotte Alley
  <li> Description: Data Loading, Storage, and File Formats

<ul>
  <li> Title: The Data Wrangling Workshop
  <li> Author: Brian Lipp, Shubhadeep Roychowdhury, Dr. Tirthajyoti Sarkar
  <li> Date: 19 October 2025
  <li> Modified By: Charlotte Alley
  <li> Description: Data Sources & Data Wrangling

In [1]:
import pandas as pd

# Reads in file with low memory as False to avoid warning and incident id as 
# index 
school_adjacent_df = pd.read_csv('school_adjacent_shootings-2025-09-21.csv', 
                                low_memory=False, index_col='incident_id')
# Ouptuts first five rows
school_adjacent_df.head(5)

,incident_date,incident_time,updated_date,state_gva,city_or_county,address,killed,injured,victims_killed,victims_injured,...,cbsatype,csa,nmcsa,necta,nmnecta,cd,sldl,sldu,ppin,state_nces
incident_id,,,,,,,,,,,,,,,,,,,,,
365165,"June 28, 2015",NaN,1435636972,Alaska,Anchorage,604 West 26th Avenue,1,0,0,0,...,1,N,N,N,N,200,18,0,A9504592,AK
461657,"December 8, 2015",NaN,1449727824,Alaska,Anchorage,3200 block of East 41st Avenue,1,0,1,0,...,1,N,N,N,N,200,17,0,A1500064,AK
284905,"January 27, 2015",NaN,1440962404,Alaska,Anchorage,6000 block of East 41st Court,2,0,2,0,...,1,N,N,N,N,200,17,0,A1500064,AK
411264,"September 8, 2015",NaN,1442453186,Alaska,Fairbanks,2000 Davis Road,1,0,0,0,...,1,N,N,N,N,200,1,0,02157038,AK
314328,"March 28, 2015",NaN,1429912332,Alaska,Anchorage,1230 West 27th Avenue,0,1,0,0,...,1,N,N,N,N,200,18,0,A9504592,AK


## Step 1: Remove Unnecessary Columns
This dataset came with a hefty 43 columns, most of which aren't intuitive to the average user. I was able to find a list identifying what each column represented, and made my selections based off of the definitions and the overall goal of the project. I ended up dropping 30 columns that were superflous to this project. For example, many of the dropped columns were identification numbers (such as 'ncessch' for National Center for Education Statistics School Identification Number) which aren't relevant to our analysis goals. An added bonus from this step is the absence of missing values in the new dataframe, evident from the Non-Null Count on the .info() output. 

In [2]:
# Drops irrelevent columns
school_adjacent_df = school_adjacent_df.drop(['ncessch', 'leaid', 'opstfips', 
                                             'cnty', 'cbsa', 'nmcbsa','csa', 
                                              'nmcsa', 'necta', 'stfip',
                                             'nmnecta', 'sldl', 'sldu', 'ppin', 
                                             'state_nces', 'updated_date', 
                                             'join_year', 'nmcnty', 'cbsatype', 
                                             'incident_time', 'date_fixed', 
                                              'city', 'zip', 'street', 'year', 
                                             'month', 'sources', 'address', 
                                              'incident_characteristics'], 
                                             axis=1)
school_adjacent_df.head()

,incident_date,state_gva,city_or_county,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested,name,locale,cd
incident_id,,,,,,,,,,,,,
365165,"June 28, 2015",Alaska,Anchorage,1,0,0,0,1,0,0,HOLY ROSARY ACADEMY,11,200
461657,"December 8, 2015",Alaska,Anchorage,1,0,1,0,0,0,1,STARBRIGHT EARLY LEARNING CENTER,11,200
284905,"January 27, 2015",Alaska,Anchorage,2,0,2,0,0,0,1,STARBRIGHT EARLY LEARNING CENTER,11,200
411264,"September 8, 2015",Alaska,Fairbanks,1,0,0,0,1,0,0,FAIRBANKS MONTESSORI ASSOCIATION,13,200
314328,"March 28, 2015",Alaska,Anchorage,0,1,0,0,0,1,5,HOLY ROSARY ACADEMY,11,200


In [3]:
# Displays new dataframe with 13 columns and 0 Non-Null values
school_adjacent_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 279235 entries, 365165 to 2824365
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   incident_date      279235 non-null  object
 1   state_gva          279235 non-null  object
 2   city_or_county     279235 non-null  object
 3   killed             279235 non-null  int64 
 4   injured            279235 non-null  int64 
 5   victims_killed     279235 non-null  int64 
 6   victims_injured    279235 non-null  int64 
 7   suspects_killed    279235 non-null  int64 
 8   suspects_injured   279235 non-null  int64 
 9   suspects_arrested  279235 non-null  int64 
 10  name               279235 non-null  object
 11  locale             279235 non-null  int64 
 12  cd                 279235 non-null  int64 
dtypes: int64(9), object(4)
memory usage: 29.8+ MB


## Step 2: Deduplicate Incidents 
Each row of this dataset represents a shooting to school match, which means that a single shooting has the potential to appear multiple times in the set depending on its proximity to nearby schools. I decided to deduplicate the data using the incident id's, and to aggregate the unique school names for each incident as a list (for multiples) or a single entry in their own column. This will allow for analysis on each shooting individually and for more accurate descriptive statistics, without overcounting cases.

In [4]:
# Outputs total number of index duplicates prior to deduplication 
school_adjacent_df.index.duplicated().sum()

np.int64(120048)

In [5]:
# Creates dictionary for looping lambda function through each column and value, 
# joins school names in name column, takes first value for all other columns 
agg_dict = {col: (lambda x, c=col: ', '.join(x.unique()) if c == 'name' else 
                  x.iloc[0]) for col in school_adjacent_df.columns}

# Deduplicates rows with identical index by concatenating unique 'name' entries
# per group, and keeping the first value for all other cells, resets index
df_dedup = (school_adjacent_df.groupby(level=0).agg(agg_dict).reset_index())  

In [6]:
df_dedup.head()

,incident_id,incident_date,state_gva,city_or_county,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested,name,locale,cd
0,111570,"March 2, 2018",Texas,Houston,1,0,1,0,0,0,0,MCREYNOLDS MIDDLE,11,4829
1,176097,"August 14, 2016",New York,New York (Manhattan),2,0,2,0,0,0,0,"PS/IS 210 21ST CENTURY ACADEMY, HAMILTON HEIGH...",11,3613
2,271945,"January 1, 2015",Illinois,Chicago,0,1,0,1,0,0,0,"Hefferan Elem School, Legal Prep Academy Chart...",11,1707
3,271947,"January 1, 2015",Illinois,Chicago,0,1,0,1,0,0,1,"Galapagos Elem Charter School, Orr Academy Hig...",11,1707
4,271951,"January 1, 2015",Nevada,Las Vegas,1,0,1,0,0,0,1,Halle Hewetson Elementary School,11,3201


In [7]:
# Outputs total number of duplicates post duplication
df_dedup['incident_id'].duplicated().sum()

np.int64(0)

In [8]:
# Drops incident id column 
df_dedup.drop('incident_id', axis=1, inplace=True)

In [9]:
df_dedup.head()

,incident_date,state_gva,city_or_county,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested,name,locale,cd
0,"March 2, 2018",Texas,Houston,1,0,1,0,0,0,0,MCREYNOLDS MIDDLE,11,4829
1,"August 14, 2016",New York,New York (Manhattan),2,0,2,0,0,0,0,"PS/IS 210 21ST CENTURY ACADEMY, HAMILTON HEIGH...",11,3613
2,"January 1, 2015",Illinois,Chicago,0,1,0,1,0,0,0,"Hefferan Elem School, Legal Prep Academy Chart...",11,1707
3,"January 1, 2015",Illinois,Chicago,0,1,0,1,0,0,1,"Galapagos Elem Charter School, Orr Academy Hig...",11,1707
4,"January 1, 2015",Nevada,Las Vegas,1,0,1,0,0,0,1,Halle Hewetson Elementary School,11,3201


## Step 3: Recode Locale
Locale is a variable that holds codes for different types of areas (such as rural, town, suburban, city). I believe this variable will be more meaningfuly analyzed and visualized with the definitions in place of the code. I sourced the definitions from this link: https://nces.ed.gov/programs/edge/Geographic/LocaleBoundaries. 

In [10]:
# Creates dictionary with locale definitions
locale_definitions = {11:'Large City', 12:'Midsize City', 13:'Small City',
    21:'Large Suburban', 22:'Midsize Suburban', 23:'Small Suburban', 31:'Fringe\
    Town', 32:'Distant Town', 33:'Remote Town', 41:'Fringe Rural', 42:'Distant\
    Rural', 43:'Remote Rural'}

# Replaces codes with their definitions
df_dedup['locale'] = df_dedup['locale'].replace(locale_definitions)
df_dedup.head()

,incident_date,state_gva,city_or_county,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested,name,locale,cd
0,"March 2, 2018",Texas,Houston,1,0,1,0,0,0,0,MCREYNOLDS MIDDLE,Large City,4829
1,"August 14, 2016",New York,New York (Manhattan),2,0,2,0,0,0,0,"PS/IS 210 21ST CENTURY ACADEMY, HAMILTON HEIGH...",Large City,3613
2,"January 1, 2015",Illinois,Chicago,0,1,0,1,0,0,0,"Hefferan Elem School, Legal Prep Academy Chart...",Large City,1707
3,"January 1, 2015",Illinois,Chicago,0,1,0,1,0,0,1,"Galapagos Elem Charter School, Orr Academy Hig...",Large City,1707
4,"January 1, 2015",Nevada,Las Vegas,1,0,1,0,0,0,1,Halle Hewetson Elementary School,Large City,3201


## Step 4: Convert Incident Date Data Type
I want to convert the incident_date column to the datetime data type using Pandas. This conversion will allow for more efficient and versatile date-time operations, such as calculating the time elapsed between shootings, filtering by date ranges, and analyzing seasonal trends in the data.

In [11]:
# Converts incident_date column to datetime format, uses coerce to create 
# Not a Time for invalid entries
df_dedup['incident_date'] = pd.to_datetime(df_dedup['incident_date'], 
                                           errors='coerce')
# Displays new data type
df_dedup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159187 entries, 0 to 159186
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   incident_date      159187 non-null  datetime64[ns]
 1   state_gva          159187 non-null  object        
 2   city_or_county     159187 non-null  object        
 3   killed             159187 non-null  int64         
 4   injured            159187 non-null  int64         
 5   victims_killed     159187 non-null  int64         
 6   victims_injured    159187 non-null  int64         
 7   suspects_killed    159187 non-null  int64         
 8   suspects_injured   159187 non-null  int64         
 9   suspects_arrested  159187 non-null  int64         
 10  name               159187 non-null  object        
 11  locale             159187 non-null  object        
 12  cd                 159187 non-null  int64         
dtypes: datetime64[ns](1), int64(8), object(4)
me

In [12]:
# Displays new format
df_dedup.head()

,incident_date,state_gva,city_or_county,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested,name,locale,cd
0,2018-03-02,Texas,Houston,1,0,1,0,0,0,0,MCREYNOLDS MIDDLE,Large City,4829
1,2016-08-14,New York,New York (Manhattan),2,0,2,0,0,0,0,"PS/IS 210 21ST CENTURY ACADEMY, HAMILTON HEIGH...",Large City,3613
2,2015-01-01,Illinois,Chicago,0,1,0,1,0,0,0,"Hefferan Elem School, Legal Prep Academy Chart...",Large City,1707
3,2015-01-01,Illinois,Chicago,0,1,0,1,0,0,1,"Galapagos Elem Charter School, Orr Academy Hig...",Large City,1707
4,2015-01-01,Nevada,Las Vegas,1,0,1,0,0,0,1,Halle Hewetson Elementary School,Large City,3201


## Step 5: Add School Name Count Column
I added a new variable representing the number of schools nearby each incident. This count converts the categorical school names variable into a numeric measure, enabling more effective analysis and visualization. By quantifying the number of nearby schools, this variable allows deeper insights into how school proximity may relate to shooting characteristics or patterns.

In [13]:
# Counts number of commas, and adds one to calculate total nearby schools
df_dedup['Nearby_Schools_Count'] = df_dedup['name'].str.count(',') + 1
df_dedup.head()

,incident_date,state_gva,city_or_county,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested,name,locale,cd,Nearby_Schools_Count
0,2018-03-02,Texas,Houston,1,0,1,0,0,0,0,MCREYNOLDS MIDDLE,Large City,4829,1
1,2016-08-14,New York,New York (Manhattan),2,0,2,0,0,0,0,"PS/IS 210 21ST CENTURY ACADEMY, HAMILTON HEIGH...",Large City,3613,3
2,2015-01-01,Illinois,Chicago,0,1,0,1,0,0,0,"Hefferan Elem School, Legal Prep Academy Chart...",Large City,1707,2
3,2015-01-01,Illinois,Chicago,0,1,0,1,0,0,1,"Galapagos Elem Charter School, Orr Academy Hig...",Large City,1707,2
4,2015-01-01,Nevada,Las Vegas,1,0,1,0,0,0,1,Halle Hewetson Elementary School,Large City,3201,1


## Step 6: Reorder and Rename Columns
I want to rearrange the dataframe for readability. I'm moving the Nearby_Schools_Count column further to the left, along with cd and locale. I also want to rename the columns so that anyone seeing it for the first time would understand what they are looking at. 

In [14]:
# Reorders columns
df_dedup = df_dedup[['incident_date', 'state_gva', 'city_or_county', 
                     'Nearby_Schools_Count', 'locale', 'cd', 'name', 'killed', 
                     'injured', 'victims_killed', 'victims_injured', 
                     'suspects_killed', 'suspects_injured', 
                     'suspects_arrested']]
df_dedup.head()

,incident_date,state_gva,city_or_county,Nearby_Schools_Count,locale,cd,name,killed,injured,victims_killed,victims_injured,suspects_killed,suspects_injured,suspects_arrested
0,2018-03-02,Texas,Houston,1,Large City,4829,MCREYNOLDS MIDDLE,1,0,1,0,0,0,0
1,2016-08-14,New York,New York (Manhattan),3,Large City,3613,"PS/IS 210 21ST CENTURY ACADEMY, HAMILTON HEIGH...",2,0,2,0,0,0,0
2,2015-01-01,Illinois,Chicago,2,Large City,1707,"Hefferan Elem School, Legal Prep Academy Chart...",0,1,0,1,0,0,0
3,2015-01-01,Illinois,Chicago,2,Large City,1707,"Galapagos Elem Charter School, Orr Academy Hig...",0,1,0,1,0,0,1
4,2015-01-01,Nevada,Las Vegas,1,Large City,3201,Halle Hewetson Elementary School,1,0,1,0,0,0,1


In [15]:
# Renames columns
df_dedup.columns = ['Incident_Date', 'State', 'City', 'Schools_Nearby', 
                    'Area_Type', 'Congressional_District', 'School Names', 
                    'Total_Killed', 'Total_Injured', 'Victims_Killed', 
                    'Victims_Injured', 'Suspects_Killed', 'Suspects_Injured', 
                    'Suspects_Arrested']
# Human readable dataset
df_dedup.head()

# Save as csv for database use 
df_dedup.to_csv('school_adjacent_shootings.csv', index=False)

## Ethical Considerations
The changes made to the data include: removing unnecessary columns, deduplicating the incidents, recoding the locale variable, converting the incident date data type, adding a count column for the nearby schools per incident, and reordering and renaming the columns. Looking at the .describe() table for the dataset, there are clearly outliers, evident by lower calculated means with such high maximums. I chose not to remove them, because they are meaningful in the context of this dataset.

The dataset contains sensitive information about shootings near schools (within 500 yards of the school's centroids), sourced primarily from the Gun Violence Archive and the U.S. Department of Education. There aren't any explicit legal or regulatory restrictions mentioned on the source's website, but because of the sensitive nature of the data, privacy and accurate reporting are essential. Considering that we are working with federal data, we need to abide by the standards they set, as well.

Each transformation completed introduces the risk of us omitting information that may be important for an accurate and whole analysis. Reordering or renaming columns, if not documented, can create misinterpretations or data confusion. Recoding variables, if not done correctly, can change their meaning and create inaccuracy. Deduplicating can be tricky, and if things don't line up exactly, the entire dataset can be compromised. The date data type conversion has the potential to introduce null values, and omitting entire columns can remove context.

There were many assumptions made in the cleaning process. By removing columns, I'm assuming that my determination of their relevancy is accurate, without running any actual tests on their potential correlations. By deduplicating, I'm assuming that each incident and school name was correctly entered into the dataset originally, and that their sources are reliable. By sourcing the definitions for the locale variable, I'm assuming the source I found was up to date and accurately relevant to my dataset. By converting the incident dates to datetime, I'm assuming the dates are correct and consistently formatted. By adding the count column, I'm assuming that the proximity metric accurately represents the exposure risk. The overarching assumption of this dataset is that school adjacent shootings are as relevant as shootings that occur inside the hallways of a school.

As previously mentioned, the dataset was compiled from the Gun Violence Archive (established in 2013 with sources in law enforcement, media reports, and public records) and the K-12 school data from the U.S. Department of Education’s National Center for Education Statistics. The data is verified by cross-checking geographic and incident information and excludes shootings without injuries or fatalities for reliability.

This dataset is publicly available from official and reputable sources. It excludes less reliable incident reports (rather than including them along with their reliability score such as in the original K-12 set). The set leaves individual names and identifiers out of it, allowing researchers to focus on the bigger picture and respecting the privacy of the individual victims.

Moving forward with the ethical implications that we have identified, I think that the proper documentation of all transformations and assumptions is key in avoiding misinterpretations. Presenting all results with the limitations that have been identified by the source, as well as remaining sensitive to the topic at hand, will be necessary in honoring the overall goal of the analysis, which is gaining insight for future prevention.

In [16]:
# Outputs descriptive statistics for final data frame
df_dedup.describe()

,Incident_Date,Schools_Nearby,Congressional_District,Total_Killed,Total_Injured,Victims_Killed,Victims_Injured,Suspects_Killed,Suspects_Injured,Suspects_Arrested
count,159187,159187.000000,159187.000000,159187.000000,159187.000000,159187.000000,159187.000000,159187.000000,159187.000000,159187.000000
mean,2020-04-06 05:07:05.544799232,1.751726,2712.640618,0.368164,0.887616,0.328632,0.851219,0.039532,0.036397,0.283396
min,2015-01-01 00:00:00,1.000000,101.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2017-11-06 00:00:00,1.000000,1701.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2020-07-30 00:00:00,1.000000,2506.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,2022-07-25 00:00:00,2.000000,3913.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,2024-12-31 00:00:00,22.000000,5600.000000,22.000000,30.000000,21.000000,30.000000,4.000000,5.000000,11.000000
std,NaN,1.451202,1492.765934,0.556462,0.788199,0.522158,0.787117,0.196974,0.200465,0.622947
